# Marginal MAP over a subset of times

`viterbi_torch_mvr_chmm` answers "what is the single most likely hidden path?".
This notebook is about a related query: **what is the most likely
assignment at a few times of interest, averaging out all other times?**

The chain × MVR product is an HMM, with state $z = (x, m)$ and with each
constraint enforced by conditioning on $\mathrm{evl}(m) = \text{True}$ at the end
of its window. The query is plain marginal MAP over the augmented chain:

$$\arg\max_{z_S}\ \sum_{z_{S^c}}\ P(z_S, z_{S^c}, y)$$

for a chosen query set of times $S$. Since the mediation part is a determinsitic function of the original chain, projecting the
augmented solution to just its $X$ part returns the marginal-MAP solution for the constrained original chain.

NOTE: This is **not** the Viterbi path restricted to $S$.
Maximizing a marginal and marginalizing a maximum are different operations, and
the notebook below shows a case where they disagree. The times outside $S$ are summed out, not deleted: they still consume a
transition, still emit if they carry an observation, and still drive every MVR
active at that time. What is dropped is only the requirement to commit to a
value there.

In [ ]:
import itertools
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.inference.viterbi_mvr import viterbi_torch_mvr_chmm
from conin.hidden_markov_model.inference.marginal_map_mvr import (
    marginal_map_torch_mvr_chmm,
)

## 1. The model

The same three-state HMM used in `MVR_viterbi.ipynb`, so the two notebooks can be
read against each other.

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

start_probs = {
    "A": 0.28,
    "B": 0.40,
    "C": 0.32,
}

transition_probs = {
    ("A", "A"): 0.34,
    ("A", "B"): 0.05,
    ("A", "C"): 0.61,
    ("B", "A"): 0.48,
    ("B", "B"): 0.06,
    ("B", "C"): 0.46,
    ("C", "A"): 0.43,
    ("C", "B"): 0.18,
    ("C", "C"): 0.39,
}

emission_probs = {
    ("A", "lo"): 0.20,
    ("A", "mid"): 0.31,
    ("A", "hi"): 0.49,
    ("B", "lo"): 0.54,
    ("B", "mid"): 0.23,
    ("B", "hi"): 0.23,
    ("C", "lo"): 0.09,
    ("C", "mid"): 0.01,
    ("C", "hi"): 0.90,
}

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
    initialize=True,
)

observed = ["hi", "mid", "lo", "lo", "lo", "lo", "lo"]
T = len(observed)

print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

`T = 7` is small enough that every one of the $3^7 = 2187$ hidden paths can be
enumerated, so every claim below is checkable by brute force.

## 2. A reference implementation

Marginal MAP is easy to state exhaustively: enumerate the paths, group them by
what they do at the query times, add up the probability inside each group, and
take the heaviest group. That is what the fast
algorithm has to reproduce.

NOTE: This block groups by hidden state
alone, which is only valid for the constraint below. In general, one should group
by the augmented state. For this particular forbidden constraint, grouping by hidden state only is ok.

In [ ]:
def path_logprob(path, obs=None):
    """Joint log probability of a hidden path and the observations."""
    obs = observed if obs is None else obs
    obs_map = obs if isinstance(obs, dict) else dict(enumerate(obs))
    repn = hmm.repn
    idx = [hmm.hidden_to_internal[h] for h in path]

    total = math.log(repn.start_vec[idx[0]])
    for t in range(1, len(idx)):
        total += math.log(repn.transition_mat[idx[t - 1]][idx[t]])
    for t, o in obs_map.items():
        total += math.log(repn.emission_mat[idx[t]][hmm.observed_to_internal[o]])

    return total


def group_masses(query_times, horizon=None, obs=None, accepts=None):
    """Total probability of every assignment to the query times."""
    horizon = T if horizon is None else horizon
    masses = {}

    for path in itertools.product(HIDDEN_STATES, repeat=horizon):
        if accepts is not None and not accepts(path):
            continue
        key = tuple(path[t] for t in query_times)
        masses[key] = masses.get(key, 0.0) + math.exp(path_logprob(path, obs))

    return masses


def brute_force_marginal_map(query_times, **kwargs):
    masses = group_masses(query_times, **kwargs)
    best = max(masses, key=masses.get)
    return list(best), math.log(masses[best])


print(f"{3 ** T} hidden paths to enumerate")

## 3. Marginal-MAP is not Viterbi

Query the first, middle and last times $S = \{0, 3, 6\}$ and compare three
things: the marginal-MAP answer, the Viterbi path restricted to those times, and
the brute-force truth.

The two disagree. Viterbi + restriction identifies the single likeliest path and reports just the query times. Marginal-MAP averages out the non-query times before maximization. 

In [ ]:
QUERY = [0, 3, 6]

model = MVR_CHMM(hidden_markov_model=hmm, constraints=[])

viterbi_path, viterbi_ll = viterbi_torch_mvr_chmm(
    model, observed, return_augmented=False, return_score=True
)
restricted = [viterbi_path[t] for t in QUERY]

mm_path, mm_score = marginal_map_torch_mvr_chmm(
    model, observed, query_times=QUERY,
    return_augmented=False, return_score=True,
)

bf_path, bf_score = brute_force_marginal_map(QUERY)

print(f"full Viterbi path        : {' '.join(viterbi_path)}   ll = {viterbi_ll:.4f}")
print(f"  ... restricted to {QUERY} : {' '.join(restricted)}")
print()
print(f"marginal MAP at {QUERY}   : {' '.join(mm_path)}   log-mass = {mm_score:.4f}")
print(f"brute force               : {' '.join(bf_path)}   log-mass = {bf_score:.4f}")
print()
print("marginal MAP matches brute force :", mm_path == bf_path
      and abs(mm_score - bf_score) < 1e-4)
print("marginal MAP == restricted Viterbi:", mm_path == restricted)

## 4. Full coverage reduces to Viterbi

If every time is queried there is nothing left to sum over, and the two
algorithms must agree exactly. In that case, `marginal_map_torch_mvr_chmm` will give a warning,
as `viterbi_torch_mvr_chmm` computes the same answer more cheaply.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    all_times = marginal_map_torch_mvr_chmm(
        model, observed, query_times=list(range(T)),
        return_augmented=False, return_score=True,
    )

print(f"viterbi      : {' '.join(viterbi_path)}   {viterbi_ll:.6f}")
print(f"marginal MAP : {' '.join(all_times[0])}   {all_times[1]:.6f}")
print()
print("identical:", all_times[0] == viterbi_path
      and abs(all_times[1] - viterbi_ll) < 1e-5)
print()
for w in caught:
    print(f"{w.category.__name__}: {w.message}")

## 5. Constrained Marginal-MAP Example

Constraints can apply to the *whole* chain, including the summed-out non-query times. A path
that violates at a non-query time still contributes nothing.

We reuse the "never visit `A`" MVR, checked against brute force with the
same acceptance test applied inside the enumeration. Note that `time_range = [3,5]`. Times `t = 4` and `t = 5` are summed out, yet the
constraint is enforced on them. We marginalize over the augmented state space for these times to maintain exactness.

In [ ]:
def forbid_state_mvr(forbidden_state, time_range=None):
    """MVR rejecting any path that visits `forbidden_state` inside its window."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={h: ("violated" if h == forbidden_state else "ok") for h in HIDDEN_STATES},
        upd={
            (m, h): ("violated" if m == "violated" or h == forbidden_state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
    )


WINDOW = [3, 5]
mvr = forbid_state_mvr("A", time_range=WINDOW)
model_c = MVR_CHMM(hidden_markov_model=hmm, constraints=[mvr])

mm_c = marginal_map_torch_mvr_chmm(
    model_c, observed, query_times=QUERY,
    return_augmented=False, return_score=True,
)
bf_c = brute_force_marginal_map(
    QUERY, accepts=lambda p: "A" not in p[WINDOW[0]:WINDOW[1] + 1]
)

print(f"marginal MAP : {' '.join(mm_c[0])}   log-mass = {mm_c[1]:.4f}")
print(f"brute force  : {' '.join(bf_c[0])}   log-mass = {bf_c[1]:.4f}")
print()
print("agree:", mm_c[0] == bf_c[0] and abs(mm_c[1] - bf_c[1]) < 1e-4)
print()
inside = [t for t in range(WINDOW[0], WINDOW[1] + 1)]
summed = [t for t in inside if t not in QUERY]
print(f"The window {WINDOW} covers t = {inside}, of which {summed} are summed out.")
print("The constraint affects the support over which marginalization is carried out -- summed-out times are constrained too.")

## 6. Sparse observations and a long horizon

`marginal_map_torch_mvr_chmm` takes the same `observed` / `time_horizon`
arguments as the Viterbi algorithm, so the query times, the observed times and
the horizon are three independent things.

This is where the algorithm earns its keep. When a stretch of time has no
observation and no active MVR, it collapses into a **power of the transition
matrix** instead of being stepped through one time at a time.

In [ ]:
BIG_T = 10_000
sparse = {0: "mid", BIG_T // 2: "hi", BIG_T - 1: "lo"}
big_query = [0, BIG_T // 2, BIG_T - 1]

t0 = time.perf_counter()
big = marginal_map_torch_mvr_chmm(
    model, sparse, time_horizon=BIG_T, query_times=big_query,
    return_augmented=False, return_score=True,
)
mm_time = time.perf_counter() - t0

t0 = time.perf_counter()
viterbi_torch_mvr_chmm(model, sparse, time_horizon=BIG_T, return_augmented=False)
vit_time = time.perf_counter() - t0

print(f"horizon      : {BIG_T} steps")
print(f"observed at  : {sorted(sparse)}")
print(f"queried at   : {big_query}")
print()
print(f"marginal MAP : {' '.join(big[0])}   log-mass = {big[1]:.4f}"
      f"   [{mm_time * 1000:.1f} ms]")
print(f"full Viterbi over all {BIG_T} steps          [{vit_time * 1000:.1f} ms]")
print()
print(f"speedup: {vit_time / mm_time:.0f}x")

Two gaps of ~5000 steps each, each collapsed into one `matrix_power` call
instead of 5000 sequential updates. The shortcut only applies when the gap has
no mediation axis and no interior observation; otherwise the interior is
propagated step by step, which is still correct, just slower.

## 7. Reading the augmented output

`return_augmented=True` reports one entry per **query time**, each tagged with
the global time it refers to, plus the mediation state of every MVR active
there.

In [ ]:
path_aug, augmented = marginal_map_torch_mvr_chmm(
    model_c, observed, query_times=QUERY
)

display(
    pd.DataFrame(
        [
            {
                "time": e["time"],
                "hidden": hmm.hidden_to_external[e["hidden_index"]],
                "MVR 0 mediation": e["mvr_states"].get(0, "-- inactive --"),
            }
            for e in augmented
        ]
    ).style.hide(axis="index").set_caption(
        f"Query times only.  MVR 0 is active on {WINDOW}."
    )
)

The MVR is active on `[3, 5]`, so it shows a mediation state at `t = 3` and
nothing at `t = 0` or `t = 6`. The rows are the query times, not the horizon —
the summed-out times have no single value to report.

## Notes

- **Marginal MAP is not Viterbi.** The states returned at the query times are in
  general not the states the joint MAP path takes there. Use
  `viterbi_torch_mvr_chmm` when the whole path is what you want.
- **The mediation state is maximized, not summed out.** The query is marginal MAP
  over the augmented chain, and the reported hidden path is the projection of it.
- Summed-out times remain part of the chain: they consume a transition, emit if
  observed, and are constrained by every MVR active there.
- The query is tractable on a chain because eliminating the summed-out run
- `query_times=None` covers the horizon and coincides with Viterbi. A warning is raised.